# 02 — Income Classifier

This notebook summarises the classifier training that `src/train_classifier.py`
runs end-to-end. Re-running `python src/train_classifier.py` from the project
root regenerates every artefact referenced here.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
metrics = json.loads((ROOT / "outputs" / "classifier_metrics.json").read_text())
metrics

## Model line-up

| Model | Use |
| --- | --- |
| **Logistic Regression** (balanced) | Interpretable baseline |
| **LightGBM** (scale_pos_weight, early stopping on val PR/AUC) | Main model |

Both use the same preprocessing pipeline:

* median imputation for numerics (tree model doesn't care; we keep it uniform for
  the pipeline),
* one-hot encoding with `min_frequency=50` to collapse rare category levels,
* sample weights are passed through to training *and* metric computation.

In [ ]:
pd.DataFrame({
    "Logistic": metrics["logreg"]["test"],
    "LightGBM": metrics["lightgbm"]["test"],
})

## Curves & confusion matrix (test set, weighted)

In [ ]:
for p in sorted((ROOT / "figures").glob("0[6-8]*.png")):
    display(Image(filename=str(p)))

## Top drivers of income

In [ ]:
imp = pd.read_csv(ROOT / "outputs" / "feature_importance.csv")
imp.head(15)

## Business interpretation

* **Weeks worked, age, and dividend income dominate** — people with full-year
  employment and investment income are far more likely to cross the \$50k line.
* **Non-filer tax status** is a strong negative signal (often retirees, kids,
  or very low income).
* **Occupation/education** add incremental signal but are dwarfed by capital
  and employment intensity.
* **Recommended decision threshold ≈ 0.29**. At this threshold, the model
  trades a little precision for big recall gains — appropriate for a marketing
  funnel that can tolerate false positives more than false negatives.
